<a href="https://colab.research.google.com/github/paulagirones/Aerogels_thermal_conductivity/blob/main/aerogels_thermal_conductivity_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerogels thermal conductivity

# DATA EXPLORATION

In [60]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('20250220_dataset.csv')

# Inspect the first rows
print("First 5 rows:")
display(df.head())

# Display information about the dataset
print("\nDataset information:")
df.info()



First 5 rows:


,Entry,Drying,Purpose,Shape,First materials,Second materials,Modification,First material class,Second materials class,Modification class,...,BET surface area m2/g,BJH pore size nm,BJH pore volume (total) cm3/g,compression elastic modulus MPa,SEM,Thermal conductivity W/(m K),Thermal method,Thermal method detail,Unnamed: 28,Unnamed: 29
0,1,SCD,Not specific,Monolith?,Soda soap,NaN,NaN,Q,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,FD,Not specific,Monolith,Polystyrene,NaN,NaN,J,NaN,NaN,...,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16559 entries, 0 to 16558
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Entry                            16559 non-null  int64  
 1   Drying                           16559 non-null  object 
 2   Purpose                          16559 non-null  object 
 3   Shape                            16559 non-null  object 
 4   First materials                  16559 non-null  object 
 5   Second materials                 6368 non-null   object 
 6   Modification                     3086 non-null   object 
 7   First material class             16559 non-null  object 
 8   Second materials class           6368 non-null   object 
 9   Modification class               3082 non-null   object 
 10  Gelation class                   16559 non-null  object 
 11  Solvent class                    16559 non-null  object 
 

# CLEANING AND PRE-PROCESSING

In [63]:
# Check Porosity type
print('Porosity type:')
print(df['Porosity'].dtype)
print(df['Porosity'].unique()[:20])

# Check SEM type
print('SEM type:')
print(df['SEM'].dtype)
print(df['SEM'].unique()[:20])


# Check Unnamed:29 type and content
print('Unnamed: 29 column content:')
print(df['Unnamed: 29'].dtype)
print(df['Unnamed: 29'].unique())


Porosity type:
object
[nan '95%' '94%' '98%' '96%' '97%' '65%' '74.38%' '80.75%' '84.19%'
 '88.06%' '87.45%' '82.64%' '78.32%' '67.97%' '32.80%' '48.20%' '53.70%'
 '59.30%' '65.90%']
SEM type:
object
[nan 'Mesoporous (nanoparticulate)' 'Macroporous (cellular)' 'Mesoporous?'
 'Macroporous (flakes)' 'Macroporous (particulate)'
 'Macroporous (fibrous)' 'Macroporous (fibrous and particulate)'
 'Mesoporous (nanofibrous)' 'Microparticles'
 'Macroporous (cellular and fibrous)'
 'Mesoporous (nanofibrous and nanoparticulate)'
 'Macroporous (cellular and flakes)' 'Macroporous (fibrous and flakes)'
 'Macroporous (nanoparticulate)' 'Macroporous (cellular and particulate)'
 'Macroporous (flakes and particulate)' 'Macroporous' 'Macroporous?'
 'Macroporous (nanofibrous)']
Unnamed: 29 column content:
object
[nan 'kokoko' 'koko' 'kokokoko' 'kokokokoko']


In [64]:
#Fix porosity type (it is numeric)
df['Porosity'] = (
    df['Porosity']
    .str.replace('%', '', regex=False)
    .astype(float)
)

display(df['Porosity'].describe())

# Remove empty and wrong columns
df = df.drop(columns=['Unnamed: 28', 'Unnamed: 29'])

,Porosity
count,7122.000000
mean,87.895411
std,14.065184
min,0.530000
25%,85.000000
50%,92.500000
75%,97.100000
max,100.000000


In [65]:
# Check missing values
missing = pd.DataFrame({
    'Missing values': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100
})
display(missing.sort_values('Missing (%)', ascending=False))

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

,Missing values,Missing (%)
Post-drying cross-linking,16064,97.010689
BJH pore size nm,15547,93.888520
BJH pore volume (total) cm3/g,14080,85.029289
Thermal conductivity W/(m K),14075,84.999094
Thermal method detail,14069,84.962860
Thermal method,14066,84.944743
compression elastic modulus MPa,13558,81.876925
Modification class,13477,81.387765
Modification,13473,81.363609
Post drying treatment,13264,80.101455


Duplicate rows: 0


In [66]:
#Rename columns
df = df.rename(columns={
    'Thermal conductivity W/(m K)': 'Thermal_conductivity',
    'Envelope density g/cm3': 'Density',
    'BET surface area m2/g': 'BET',
    'BJH pore size nm': 'Pore_size',
    'BJH pore volume (total) cm3/g': 'Pore_volume',
    'compression elastic modulus MPa': 'Elastic_modulus'
})

# Remove samples without a measured thermal conductivity (our target)
df = df.dropna(subset=['Thermal_conductivity'])

# Replace missing categorical values with "Unknown"
categorical_columns = df.select_dtypes(include='object').columns
df[categorical_columns] = df[categorical_columns].fillna('Unknown')

#Dataset after cleaning
print(f"Dataset after cleaning: {df.shape}")

missing = pd.DataFrame({
    'Missing values': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100
})
display(missing.sort_values('Missing (%)', ascending=False))

Dataset after cleaning: (2484, 28)


,Missing values,Missing (%)
Pore_size,2324,93.558776
Pore_volume,2166,87.198068
Elastic_modulus,1657,66.706924
BET,1525,61.392915
Porosity,1079,43.438003
Density,274,11.030596
First materials,0,0.000000
Second materials,0,0.000000
Entry,0,0.000000
Drying,0,0.000000


In [71]:
# Display categories and number of unique values
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(f"Number of categories: {df[col].nunique()}")
    print(df[col].unique()[:20])


--- Drying ---
Number of categories: 5
['SCD' 'FD' 'APD' 'VD' 'Foaming']

--- Purpose ---
Number of categories: 28
['Thermal insulation' 'Not specific'
 'Sound insulation, thermal insulation' 'Adsorbent' 'Filters' 'Separation'
 'Adsorbent (gas)' 'Thermoelectric devices' 'Steam generation'
 'Separation, thermal insulation' 'Adsorbent, thermal insulation'
 'PCM support' 'Separation, thermal Insulation' 'Sound insulation'
 'Filters, thermal insulation' 'Sensors, thermal insulation'
 'Optoelectronic devices' 'Shield, thermal insulation' 'Packaging'
 'Adsorbent (gas), sensors']

--- Shape ---
Number of categories: 11
['Monolith' 'Fibers, monolith' 'Monolith ' 'Sheet' 'Fibers'
 'Monolith, sheet' 'Monolith (3D)' 'Beads, monolith' 'Beads' 'FIbers'
 'Coating']

--- First materials ---
Number of categories: 90
['RF' 'Phenolic resin' 'Polyisocyanurate' 'Polyurethane'
 'Polydicyclopentadiene' 'Poly(urea-urethane)' 'Aramid fibers' 'Polyimide'
 'Lignocellulose' 'Cellulose' 'Cellulose nanofibers' 'P

In [73]:

# Remove leading and trailing whitespace from categorical variables

categorical_columns = df.select_dtypes(include='object').columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

# Display again categories and number of unique values
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(f"Number of categories: {df[col].nunique()}")
    print(df[col].unique()[:20])




--- Drying ---
Number of categories: 5
['SCD' 'FD' 'APD' 'VD' 'Foaming']

--- Purpose ---
Number of categories: 28
['Thermal insulation' 'Not specific'
 'Sound insulation, thermal insulation' 'Adsorbent' 'Filters' 'Separation'
 'Adsorbent (gas)' 'Thermoelectric devices' 'Steam generation'
 'Separation, thermal insulation' 'Adsorbent, thermal insulation'
 'PCM support' 'Separation, thermal Insulation' 'Sound insulation'
 'Filters, thermal insulation' 'Sensors, thermal insulation'
 'Optoelectronic devices' 'Shield, thermal insulation' 'Packaging'
 'Adsorbent (gas), sensors']

--- Shape ---
Number of categories: 10
['Monolith' 'Fibers, monolith' 'Sheet' 'Fibers' 'Monolith, sheet'
 'Monolith (3D)' 'Beads, monolith' 'Beads' 'FIbers' 'Coating']

--- First materials ---
Number of categories: 90
['RF' 'Phenolic resin' 'Polyisocyanurate' 'Polyurethane'
 'Polydicyclopentadiene' 'Poly(urea-urethane)' 'Aramid fibers' 'Polyimide'
 'Lignocellulose' 'Cellulose' 'Cellulose nanofibers' 'Polyurea' 'Pec

First material class,A,B,C,D,E,F,G,H,I,J,K,L,M,N,Q,R
First materials,,,,,,,,,,,,,,,,
Agar,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Agarose,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Alginate,85,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Aramid fibers,0,0,0,0,0,0,0,0,0,0,103,0,0,0,0,0
Bacterial cellulose,0,0,0,0,0,0,0,0,0,0,33,0,0,0,0,0
COF,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,0
Carbon fibers,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Carbon nanofibers,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0
Carboxymethyl cellulose,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
